In [ ]:
#Imports & config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent / "src"))

from utils import load_dataset, ensure_sorted, train_test_split_time_series
from feature_engineering import add_lag_features, add_rolling_features, add_time_features
from evaluation import evaluate_forecast, print_evaluation


#2 — Paths--

In [ ]:
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

MERGED_PATH = PROCESSED_DIR / "demand_temperature_half_hourly.csv"


3 — Load & inspect data

In [ ]:
df = load_dataset(MERGED_PATH)
df = ensure_sorted(df)

df.head(), df.info()


4 — Basic plot

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df['timestamp'], df['demand'], label='Demand')
plt.xlabel("Time")
plt.ylabel("Demand (MW)")
plt.title("NESO Half-hourly Demand")
plt.legend()
plt.tight_layout()
plt.show()


5 — Feature engineering

In [ ]:
df_fe = df.copy()

df_fe = add_lag_features(df_fe, column='demand', lags=[1, 48, 96])
df_fe = add_rolling_features(df_fe, column='demand', windows=[48, 96, 336])
df_fe = add_time_features(df_fe)

df_fe = df_fe.dropna().reset_index(drop=True)
df_fe.head()


6 — Train/test split

In [ ]:
train, test = train_test_split_time_series(df_fe, test_size=0.1)

features = [c for c in df_fe.columns if c not in ['timestamp', 'demand']]
X_train, y_train = train[features], train['demand']
X_test, y_test = test[features], test['demand']


7 — Simple baseline model (e.g. last value)

In [ ]:
y_pred_naive = test['demand'].shift(1).fillna(method='bfill')

metrics_naive = evaluate_forecast(y_test, y_pred_naive)
print("Naive (t-1) baseline:")
print_evaluation(metrics_naive)


8 — Plot forecast vs actual

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(test['timestamp'], y_test, label='Actual')
plt.plot(test['timestamp'], y_pred_naive, label='Naive forecast', alpha=0.7)
plt.legend()
plt.title("Baseline Forecast vs Actual")
plt.tight_layout()
plt.show()
